In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from getpass import getpass

username = "postgres"
password = getpass("Enter your PostgreSQL password: ")

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=username,
    password=password,
    host="localhost",
    port=5432,
    database="olist_analytics"
)

engine = create_engine(connection_url)

# Test the connection
with engine.connect() as connection:
    result = connection.exec_driver_sql(
        "SELECT current_database();"
    )
    print("Connected to:", result.scalar())

In [ ]:
import pandas as pd
from pathlib import Path

# Locate your existing cleaned master dataset
csv_path = Path("../cleaned_data/olist_master_dataset.csv")

# Load the dataset
df = pd.read_csv(csv_path)

# Verify the data
print("Dataset shape:", df.shape)
print("Number of columns:", len(df.columns))
print("\nFirst five rows:")
display(df.head())

In [ ]:
from sqlalchemy import text

# Create the table and import the dataset
df.to_sql(
    name="olist_master",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=2000
)

print("Dataset imported successfully!")

In [ ]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM olist_master")
    )
    print("Rows imported:", result.scalar())

In [ ]:
from sqlalchemy import text

query = text("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'olist_master'
    ORDER BY ordinal_position;
""")

with engine.connect() as connection:
    columns = connection.execute(query)

    for row in columns:
        print(row)

In [ ]:

from sqlalchemy import text
import pandas as pd

query = text("""
    SELECT
        ROUND(SUM(item_total_value)::numeric, 2) AS total_sales,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(
            (SUM(item_total_value) / COUNT(DISTINCT order_id))::numeric,
            2
        ) AS average_order_value
    FROM olist_master;
""")

with engine.connect() as connection:
    result = pd.read_sql(query, connection)

display(result)


In [ ]:

query = text("""
    SELECT
        COALESCE(
            product_category_name_english,
            'Unknown'
        ) AS product_category,
        ROUND(SUM(item_total_value)::numeric, 2) AS total_sales,
        COUNT(DISTINCT order_id) AS total_orders,
        ROUND(
            AVG(item_total_value)::numeric,
            2
        ) AS average_item_value
    FROM olist_master
    GROUP BY product_category_name_english
    ORDER BY total_sales DESC
    LIMIT 10;
""")

with engine.connect() as connection:
    result = pd.read_sql(query, connection)

display(result)


In [ ]:

query = text("""
    SELECT
        TO_CHAR(
            TO_TIMESTAMP(
                order_purchase_timestamp,
                'YYYY-MM-DD HH24:MI:SS'
            ),
            'YYYY-MM'
        ) AS sales_month,
        ROUND(SUM(item_total_value)::numeric, 2) AS total_sales,
        COUNT(DISTINCT order_id) AS total_orders
    FROM olist_master
    WHERE order_purchase_timestamp IS NOT NULL
    GROUP BY sales_month
    ORDER BY sales_month;
""")

with engine.connect() as connection:
    monthly_sales = pd.read_sql(query, connection)

display(monthly_sales)


In [ ]:

import matplotlib.pyplot as plt
import pandas as pd

monthly_sales["sales_month"] = pd.to_datetime(
    monthly_sales["sales_month"],
    format="%Y-%m"
)

plt.figure(figsize=(14, 6))

plt.plot(
    monthly_sales["sales_month"],
    monthly_sales["total_sales"],
    marker="o"
)

plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Total Sales (R$)")
plt.xticks(rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

query = text("""
    SELECT
        delivery_status,
        COUNT(DISTINCT order_id) AS total_orders
    FROM olist_master
    WHERE order_status = 'delivered'
      AND delivery_status IS NOT NULL
    GROUP BY delivery_status
    ORDER BY total_orders DESC;
""")

with engine.connect() as connection:
    delivery_summary = pd.read_sql(query, connection)

display(delivery_summary)


In [ ]:

query = text("""
    SELECT
        delivery_status,
        COUNT(DISTINCT order_id) AS total_orders
    FROM olist_master
    WHERE order_status = 'delivered'
      AND delivery_status IS NOT NULL
    GROUP BY delivery_status
    ORDER BY total_orders DESC;
""")

with engine.connect() as connection:
    delivery_summary = pd.read_sql(query, connection)

display(delivery_summary)
